# Section 3: Pooled training data and baselines

Builds the training data for the toxicity classifier (package 4) from four labelled sources and tunes the logistic-regression baselines on it.

Pipeline: load and map labels → cap each source → clean → remove duplicates → stratified train / validation split → Bayesian-tuned logistic regressions → validation scores.

Toxicity definition :

> a comment is toxic if it insults, belittles, harasses, threatens or attacks a person or group (including third parties not in the discussion), or is clearly abusive or hostile. Negative emotion, strong disagreement, criticism of a workplace and swearing that is not aimed at anyone are not enough on their own.



The external labels are not rewritten to fit this definition. They follow the proposal's mapping below, so the sources still differ in what they call toxic (for example, Dota labels most swearing as toxic).

## 1. Setup

In [ ]:
# Add the repo root to Python's search path
import sys
sys.path.append('..')

from functions.pooled_data import prepare_pooled_data

import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (ConfusionMatrixDisplay, average_precision_score, f1_score,
                             precision_recall_curve, precision_score, recall_score)

RANDOM_STATE = 42
CAP_PER_SOURCE = 10_000
TEST_SIZE = 0.2
THRESHOLD = 0.5
DATA_DIR = '../dataset'
OUT_DIR = '../dataset/pooled'
FIG_DIR = os.path.join(OUT_DIR, 'figures')

import sys
print(sys.executable)

## 2. Load, clean and split

Map the four sources to binary labels according to the proposal.

| Source | File in dataset/ | Toxic (1) if |
|---|---|---|
| youtube | youtoxic_english_1000.csv | IsToxic is true |
| wikipedia | jigsaw-toxic-comment-classification-challenge/train.csv.zip | any of the six toxicity labels is 1 |
| news | jigsaw-unintended-bias-in-toxicity-classification/train.csv | target ≥ 0.5 |
| dota | gosu-ai-english-dota-chat/tagged-data.csv | target is 1 (mild) or 2 (strong) |

Cap Wikipedia and news at 10,000 rows each, preserving their class proportions;
retain all YouTube and Dota rows. Lower-case the text and apply cleanup_text.
Remove empty texts and duplicates across sources: keep one copy when labels
agree and remove all copies when they conflict. Then split 80/20, stratified
by source and label.

In [ ]:
train_df, val_df, audit = prepare_pooled_data(
    DATA_DIR, cap_per_source=CAP_PER_SOURCE, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
pooled = pd.concat([train_df, val_df], ignore_index=True)

print('Removed rows:')
print(audit['removed'].groupby(['source', 'reason']).size().to_string())

stages = ['raw', 'sampled', 'cleaned', 'deduplicated', 'train', 'val']
audit['summary'].pivot(index='source', columns='stage', values='rows')[stages]

## 3. Save

Export the fixed splits for package 4 to dataset/pooled/, together with
the source counts and removal audit. Both CSVs contain orig_id, source,
text_raw, text and label. The manifest records settings and file hashes
for version checks. These exports are ignored by Git.

In [ ]:
os.makedirs(FIG_DIR, exist_ok=True)
train_df.to_csv(os.path.join(OUT_DIR, 'pooled_train.csv'), index=False)
val_df.to_csv(os.path.join(OUT_DIR, 'pooled_val.csv'), index=False)
audit['summary'].to_csv(os.path.join(OUT_DIR, 'data_summary.csv'), index=False)
audit['removed'].to_csv(os.path.join(OUT_DIR, 'removed_rows.csv'), index=False)

manifest = audit['manifest']
with open(os.path.join(OUT_DIR, 'manifest.json'), 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)
print(f'Saved {len(train_df):,} training and {len(val_df):,} validation rows to {OUT_DIR}')

## 4. Explore the pooled data

Compare source sizes, class balance and comment lengths. The table reports
mean, median, p90, maximum word count and the proportion exceeding 80 words.

In [ ]:
lengths = pooled['text'].str.split().str.len()
counts = pooled.groupby('source')['label'].agg(rows='size', toxic='sum')

length_summary = lengths.groupby(pooled['source']).describe(percentiles=[0.5, 0.9])
length_summary = length_summary[['mean', '50%', '90%', 'max']].rename(columns={'50%': 'median', '90%': 'p90'})
length_summary['over_80_words'] = lengths.gt(80).groupby(pooled['source']).mean()
length_summary.to_csv(os.path.join(OUT_DIR, 'text_length_summary.csv'))
display(length_summary.round(2))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(counts.index, counts['rows'] - counts['toxic'], label='non-toxic')
ax[0].bar(counts.index, counts['toxic'], bottom=counts['rows'] - counts['toxic'], label='toxic')
ax[0].set(title='Rows per source', ylabel='Rows')
ax[0].legend()
ax[1].boxplot([lengths[pooled['source'] == s] for s in counts.index], tick_labels=counts.index, showfliers=False)
ax[1].set(title='Comment length (outliers hidden)', ylabel='Words')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'source_and_length.png'), dpi=160)
plt.show()

## 5. Joint TF-IDF and LR tuning
Tune standard and balanced LR separately on `pooled_train` only, with the same protocol as the Bi-LSTM search in notebook 4 so that the comparison in 4b is like for like.

- **Search:** Bayesian (scikit-optimize ask/tell), 30 trials with 10 random starts per variant, seed 42. The space is `C` (log-uniform, 0.01 to 100), the n-gram range (unigrams or unigrams + bigrams) and `min_df` (integer, 1 to 5). TF-IDF is refitted inside each training fold.
- **Folds and metric:** five stratified folds (shuffled, seed 42, identical to the Bi-LSTM and to both LR variants), mean CV average precision as the objective.
- **Selection (`stable`):** among trials whose mean AP is within one fold SD (of the best trial) of the best mean, pick the one with the lowest sample SD across folds. This is the rule used for the Bi-LSTM.

Refit each selected pipeline on all training rows, then evaluate on `pooled_val` at the fixed 0.5 threshold. No Reddit data is used.

These are the only logistic regressions saved: the tuned pair replaces the earlier fixed-settings baselines in `../models/baselines/`, which notebook 4b loads. The earlier untuned scores (`baseline_metrics.csv`, `loso_metrics.csv`, `transfer_comparison.csv` in `../dataset/pooled/`) are kept as a frozen "before tuning" reference and are no longer regenerated by this notebook.

This tuning was added after the first validation results were seen. Fix the search space and selection rule before running; do not revise them in response to hold-out scores. The earlier grid search (`tune_logistic_regressions`, `selection='best'`) is still in `functions/lr_tuning.py` but is not the protocol used here.

The two searches require 302 fits in total and may take several minutes; enable `RUN_LR_TUNING` only when ready. Saving requires the separate `SAVE_TUNED_LR` switch and replaces the two LR files loaded by notebook 4b, so obtain team approval first. Restart notebook 4b and rerun its evaluation after saving. Restart the kernel after editing shared functions.

In [ ]:
from functions.lr_tuning import bayes_tune_logistic_regressions
from functions.metrics import metrics_by_group

RUN_LR_TUNING = False  # set to True to run the tuning; False to skip and use the saved models
SAVE_TUNED_LR = False  # explicit permission to replace the saved LR models
LR_SEARCH = {'n_iter': 30, 'n_initial_points': 10, 'selection': 'stable'}  # same as the Bi-LSTM search in notebook 4

if SAVE_TUNED_LR and not RUN_LR_TUNING:
    raise ValueError('Enable RUN_LR_TUNING too, so stale in-memory results cannot be saved.')

if RUN_LR_TUNING:
    tuned_lr = bayes_tune_logistic_regressions(
        train_df['text'], train_df['label'], n_splits=5, random_state=RANDOM_STATE,
        n_jobs=1, **LR_SEARCH,
    )
    display(tuned_lr['cv_results'].round(4))
    print('Selected parameters:', tuned_lr['config']['selected_params'])
    tuned_tables = []
    for name, model in tuned_lr['models'].items():
        proba = model.predict_proba(val_df['text'])[:, 1]
        table = metrics_by_group(val_df['label'], proba, val_df['source'], threshold=0.5).reset_index()
        table.insert(0, 'model', name)
        table['threshold'] = 0.5
        tuned_tables.append(table)
    tuned_lr_metrics = pd.concat(tuned_tables, ignore_index=True)
    display(tuned_lr_metrics.round(4))

    if SAVE_TUNED_LR:
        import hashlib
        import joblib

        # Verify that the manifest identifies the exact training/validation frames used.
        for split, frame in [('train', train_df), ('val', val_df)]:
            digest = hashlib.sha256(frame.to_csv(index=False, lineterminator='\n').encode()).hexdigest()
            if digest != manifest['split_sha256'][split]:
                raise ValueError(f'{split} data differs from the manifest; refusing to save.')
        model_dir = '../models/baselines'
        os.makedirs(model_dir, exist_ok=True)
        for name, model in tuned_lr['models'].items():
            joblib.dump(model, os.path.join(model_dir, f'{name}.joblib'))
        tuned_lr['cv_results'].to_csv(os.path.join(model_dir, 'lr_tuning_cv.csv'), index=False)
        tuned_lr_metrics.to_csv(os.path.join(model_dir, 'lr_tuning_validation.csv'), index=False)
        with open(os.path.join(model_dir, 'baselines_manifest.json'), 'w', encoding='utf-8') as f:
            json.dump({'train_split_sha256': manifest['split_sha256']['train'],
                       'val_split_sha256': manifest['split_sha256']['val'],
                       'random_state': RANDOM_STATE, 'threshold': 0.5,
                       'config': tuned_lr['config']}, f, indent=2)
        print('Saved tuned LR models. Restart and run notebook 4b to compare them with the Bi-LSTMs.')
else:
    print('LR tuning skipped. Set RUN_LR_TUNING=True to run train-only CV.')

## 6. Validation performance
The saved tuned models are scored on the held-out validation split, which was never used for training or tuning. It contains comments from the same four sources as the training data, so these scores are in-domain and optimistic about r/jobs; the zero-shot test on r/jobs and the comparison with the Bi-LSTMs are in `4b_LSTM_Logistic_Comparison.ipynb`. The models are loaded from `../models/baselines/`, so this section works after a fresh tuning run or from the saved files, and it refuses files fitted on a different training split or not produced by the Bayesian search. The three parts below mirror the validation section of notebook 4: metrics and confusion matrices at the fixed threshold, precision-recall curves, and a table of other cutoffs.

### Metrics and confusion matrices
The table reports the proposal's metrics at the fixed 0.5 threshold. Precision and recall are for the toxic class, and macro F1 averages both classes. PR-AUC is the average precision of the predicted probabilities, so it does not depend on the threshold.

In [ ]:
baseline_dir = '../models/baselines'
with open(os.path.join(baseline_dir, 'baselines_manifest.json'), encoding='utf-8') as f:
    baselines_manifest = json.load(f)
if baselines_manifest['train_split_sha256'] != manifest['split_sha256']['train']:
    raise ValueError('The saved logistic regressions were fitted on a different training split: re-run the tuning.')
if baselines_manifest['config'].get('search') != 'bayes':
    raise ValueError('The saved logistic regressions are not the Bayesian-tuned models: run and save the tuning above.')

lr_models = {
    'LR (standard)': joblib.load(os.path.join(baseline_dir, 'tfidf_logreg.joblib')),
    'LR (balanced)': joblib.load(os.path.join(baseline_dir, 'tfidf_logreg_balanced.joblib')),
}
lr_colors = {'LR (standard)': '#7f7f7f', 'LR (balanced)': '#8172B3'}  # as in notebook 4b

actual = val_df['label'].to_numpy(dtype=int)
val_proba_by_model = {name: model.predict_proba(val_df['text'])[:, 1] for name, model in lr_models.items()}
val_pred_by_model = {name: (p >= THRESHOLD).astype(int) for name, p in val_proba_by_model.items()}

In [ ]:
# precision, recall and macro F1 at the fixed threshold; PR-AUC (average precision) from the probabilities
val_metrics = pd.DataFrame([{
    'n_comments': len(actual),
    'threshold': THRESHOLD,
    'precision': precision_score(actual, pred, zero_division=0),
    'recall': recall_score(actual, pred, zero_division=0),
    'macro_f1': f1_score(actual, pred, average='macro', zero_division=0),
    'pr_auc': average_precision_score(actual, val_proba_by_model[name]),
} for name, pred in val_pred_by_model.items()], index=list(lr_models))
display(val_metrics.round(4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, pred) in zip(axes, val_pred_by_model.items()):
    ConfusionMatrixDisplay.from_predictions(
        actual, pred, display_labels=['Non-toxic', 'Toxic'], cmap='Blues', colorbar=False, ax=ax,
    )
    ax.set_title(f'{name}: validation confusion matrix')
plt.tight_layout()
plt.show()

### Precision-recall curves
Precision (the share of flagged comments that are toxic) is plotted against recall (the share of toxic comments caught) as the cutoff moves from 1 to 0, for both models on the same axes. The area under each curve is the PR-AUC, which summarises how well a model ranks toxic above clean comments whatever the cutoff, and it is the metric the tuning maximises. The dashed line is the toxic share of the validation split, which is the PR-AUC of random ranking, and the dot on each curve marks the operating point at the fixed threshold.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for name in lr_models:
    proba, pred = val_proba_by_model[name], val_pred_by_model[name]
    precision, recall, _ = precision_recall_curve(actual, proba)
    ax.step(recall, precision, where='post', color=lr_colors[name], linewidth=2,
            label=f'{name} (PR-AUC = {average_precision_score(actual, proba):.3f})')
    # operating point at the fixed threshold
    op_recall = ((pred == 1) & (actual == 1)).sum() / (actual == 1).sum()
    op_precision = ((pred == 1) & (actual == 1)).sum() / max((pred == 1).sum(), 1)
    ax.scatter([op_recall], [op_precision], color=lr_colors[name], edgecolors='black', zorder=3, s=70,
               label=f'{name} threshold = {THRESHOLD:g}')
ax.axhline(actual.mean(), color='grey', linestyle='--', label=f'Chance (PR-AUC = {actual.mean():.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.05)
ax.set_title('Validation precision-recall curves')
ax.legend(loc='lower left', fontsize=8)
plt.tight_layout()
plt.show()

### Cutoff trade-off
The fixed cutoff is a choice, so this table shows what other cutoffs would do on the validation split: the recall, the precision, the false positive rate and the share of comments flagged. Lowering the cutoff catches more toxic comments but flags more clean ones. It is for reference and does not change the threshold. Unlike the Bi-LSTM table in notebook 4, there are no out-of-fold columns, because the tuning does not keep out-of-fold predictions. The validation toxic share is about 15%, so precision here is in-domain; on r/jobs the share is unknown and precision would change with it.

In [ ]:
# recall = share of toxic comments caught | precision = share of flagged comments that are toxic
# fpr = share of non-toxic comments flagged | flagged = share of all comments flagged
cutoffs = [0.5, 0.4, 0.3, 0.25, 0.2, 0.15, 0.1, 0.05]
for name, proba in val_proba_by_model.items():
    rows = []
    for cutoff in cutoffs:
        pred = (proba >= cutoff).astype(int)
        rows.append({
            'threshold': cutoff,
            'recall': recall_score(actual, pred, zero_division=0),
            'precision': precision_score(actual, pred, zero_division=0),
            'fpr': ((pred == 1) & (actual == 0)).sum() / (actual == 0).sum(),
            'flagged': pred.mean(),
        })
    print(name)
    display(pd.DataFrame(rows).set_index('threshold').round(2))

## 7. Handoff to package 4

- Load pooled_train.csv and pooled_val.csv with keep_default_na=False and
  orig_id as a string. Use text as input and label as the target.
- Text is already lower-cased and cleaned. Apply the same pipeline to new
  comments, followed by the saved tokenizer and padding settings.
- Fit the tokenizer and run cross-validation and hyperparameter tuning only within
  pooled_train. The decision threshold is fixed at 0.5. Reserve pooled_val and the final r/jobs test set for evaluation.
- Compare the Bi-LSTM and baselines on the same validation split using the
  shared precision, recall, Macro F1 and AP functions.
- Regenerate the split and retrain if data, mappings or cleaning change.
  Check split_sha256 in manifest.json to match data versions.